# Simple Average Ensemble Model

This notebook creates a **simple average ensemble** using predictions from:
1. **exp_8_hybrid_charcnn_mlp** - Hybrid CharCNN + MLP (Neural Network)
2. **exp_4_all_xgboost_optuna** - XGBoost with Optuna tuning
3. **exp_2_tfidf_ridge** - Ridge Classifier with TF-IDF
4. **exp_12_deberta** - DeBERTa-v3 Transformer Model

## Ensemble Architecture

**Simple Averaging:**
- Load pre-trained models from experiment directories
- Generate out-of-fold predictions on train data (for evaluation)
- Generate predictions on test data
- Average all predictions equally

All models use the **same fold splits** (StratifiedKFold with random_state=42), ensuring proper alignment of out-of-fold predictions.

## Important Notes

**Hybrid Model Requirements:**
- The Keras hybrid model (exp_8) requires both character sequences and MLP features
- Loads: model, preprocessor (dict with 'preprocessor' and 'scaler'), and char tokenizer
- Preprocessing: text → char sequences (pad to 1000) + TF-IDF+structured → scaled features
- Model expects: `[char_input, mlp_input]`

**DeBERTa Model Requirements:**
- DeBERTa-v3 transformer model (exp_12)
- Loads: transformer model from saved checkpoint
- Preprocessing: text → tokenized sequences (max_length=512)
- Uses PyTorch/Transformers library

In [1]:
use_drive = False

# uncomment if running on colab
# from google.colab import drive
# drive.mount('/content/drive')
# use_drive = True

In [2]:
import pandas as pd
import numpy as np
import pickle
import json
import os
from datetime import datetime

# Sklearn tools
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, log_loss
)

# Scipy for sigmoid conversion
from scipy.special import expit

# Deep learning
import tensorflow as tf
from tensorflow import keras

# PyTorch and Transformers for DeBERTa
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

TensorFlow version: 2.20.0
Keras version: 3.12.0
PyTorch version: 2.9.0
CUDA available: False


## 1. Configuration and Setup

In [3]:
if use_drive:
    DRIVE_PATH = '/content/drive/MyDrive/Kaggle'
    DATA_PATH = os.path.join(DRIVE_PATH, "data/")
    EXPERIMENT_PATH = os.path.join(DRIVE_PATH, "experiments/")
else:
    DATA_PATH = "data/"
    EXPERIMENT_PATH = "experiments/"

ENSEMBLE_NAME = "exp_10a_simple_ensemble"
ENSEMBLE_DIR = os.path.join(EXPERIMENT_PATH, ENSEMBLE_NAME)

# Create ensemble directory
os.makedirs(ENSEMBLE_DIR, exist_ok=True)
print(f"Ensemble directory: {ENSEMBLE_DIR}")

# Base model configurations
BASE_MODELS = {
    'deberta': {
        'path': os.path.join(EXPERIMENT_PATH, 'exp_12_deberta') if use_drive else 'experiments/exp_12_deberta',
        'type': 'transformer',
        'model_prefix': 'models/best_model_fold_',
        'tokenizer_name': 'microsoft/deberta-v3-base',
        'max_length': 512,
        'batch_size': 32
    },
    'xgboost_optuna': {
        'path': os.path.join(DRIVE_PATH, 'experiments_old/exp_4_all_xgboost') if use_drive else 'experiments_old/exp_4_all_xgboost',
        'type': 'sklearn',
        'pipeline_prefix': 'pipeline_fold_',
        'pipeline_suffix': '_optuna.pkl'
    },
    'ridge': {
        'path': os.path.join(DRIVE_PATH, 'experiments_old/exp_2_tfidf_ridge') if use_drive else 'experiments_old/exp_2_tfidf_ridge',
        'type': 'sklearn',
        'pipeline_prefix': 'pipeline_fold_',
        'pipeline_suffix': '.pkl'
    },
    'hybrid_charcnn_mlp': {
        'path': os.path.join(DRIVE_PATH, 'experiments_old/exp_8_hybrid_charcnn_mlp') if use_drive else 'experiments_old/exp_8_hybrid_charcnn_mlp',
        'type': 'keras',
        'model_prefix': 'models/model_fold_',
        'preprocessor_prefix': 'models/preprocessor_fold_',
        'model_suffix': '.keras',
        'preprocessor_suffix': '.pkl',
        'tokenizer_file': 'tokenizer.json'  # Char tokenizer for CharCNN
    }
}

# Cross-validation setup (MUST match base models)
N_FOLDS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"\nData path: {DATA_PATH}")
print(f"Experiment path: {EXPERIMENT_PATH}")
print(f"Base models: {list(BASE_MODELS.keys())}")
print(f"Cross-validation: {N_FOLDS} folds, random_state={RANDOM_STATE}")

Ensemble directory: experiments/exp_10a_simple_ensemble

Base models: ['deberta', 'xgboost_optuna', 'ridge', 'hybrid_charcnn_mlp']
Cross-validation: 5 folds, random_state=42


## 2. Load Data

In [4]:
# Load training and test data
train = pd.read_csv(os.path.join(DATA_PATH, "train_w_struct.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test_w_struct.csv"))

# Deduplicate rows on text column
train = train.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Extract target and ids
y = train['label'].values
test_ids = test['id'].values

print(f"\nTarget distribution:")
print(f"  Class 0: {np.sum(y == 0)} ({np.sum(y == 0) / len(y) * 100:.2f}%)")
print(f"  Class 1: {np.sum(y == 1)} ({np.sum(y == 1) / len(y) * 100:.2f}%)")

Train shape: (4212, 36)
Test shape: (1178, 35)

Target distribution:
  Class 0: 2733 (64.89%)
  Class 1: 1479 (35.11%)


## 3. Load Base Model Metrics

First, let's load and display the performance of each base model.

In [5]:
# Load metrics for each base model
base_model_metrics = {}

for model_name, config in BASE_MODELS.items():
    metrics_path = os.path.join(config['path'], 'metrics.json')
    if model_name == 'xgboost_optuna':
        metrics_path = os.path.join(config['path'], 'metrics_optuna.json')
    
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
        base_model_metrics[model_name] = metrics
        
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    print(f"Type: {metrics['model_type']}")
    if 'vectorizer' in metrics:
        print(f"Vectorizer: {metrics['vectorizer']}")
    elif 'transformer_model' in metrics:
        print(f"Transformer: {metrics['transformer_model']}")
    print(f"\nAverage Metrics (across {metrics['n_folds']} folds):")
    for metric, value in metrics['average_metrics'].items():
        if '_std' not in metric:
            std_value = metrics['average_metrics'].get(f"{metric}_std", 0)
            print(f"  {metric:12s}: {value:.6f} ± {std_value:.6f}")


Model: deberta
Type: DeBERTa-v3
Transformer: microsoft/deberta-v3-base

Average Metrics (across 5 folds):
  accuracy    : 0.984566 ± 0.006774
  precision   : 0.966565 ± 0.015932
  recall      : 0.990531 ± 0.006506
  f1          : 0.978348 ± 0.009313
  roc_auc     : 0.998762 ± 0.001340
  log_loss    : 0.071161 ± 0.039553

Model: xgboost_optuna
Type: XGBoost
Vectorizer: TF-IDF+TruncatedSVD

Average Metrics (across 5 folds):
  accuracy    : 0.969654 ± 0.006380
  precision   : 0.963915 ± 0.009998
  recall      : 0.938463 ± 0.013117
  f1          : 0.950987 ± 0.010372
  roc_auc     : 0.995114 ± 0.001975
  log_loss    : 0.086210 ± 0.021418

Model: ridge
Type: Ridge Classifier
Vectorizer: TF-IDF

Average Metrics (across 5 folds):
  accuracy    : 0.974744 ± 0.006122
  precision   : 0.985008 ± 0.006408
  recall      : 0.933733 ± 0.016164
  f1          : 0.958639 ± 0.010280
  roc_auc     : 0.997613 ± 0.000646

Model: hybrid_charcnn_mlp
Type: Hybrid CharCNN + MLP (Keras)
Vectorizer: CharTokenize

## 4. Generate Out-of-Fold Predictions (Train)

For the stacked ensemble, we need out-of-fold predictions from each base model on the training data. Since all models use the same fold splits, we can load predictions from each fold.

In [6]:
# Apply sklearn compatibility fix once at module level
import sys
from sklearn.compose import _column_transformer

if not hasattr(_column_transformer, '_RemainderColsList'):
    class _RemainderColsList(list):
        pass
    _column_transformer._RemainderColsList = _RemainderColsList
    sys.modules['sklearn.compose._column_transformer']._RemainderColsList = _RemainderColsList
    print("✓ Applied sklearn compatibility fix for _RemainderColsList")


def load_char_tokenizer(tokenizer_path):
    """Load character tokenizer from JSON (shared across all folds)."""
    with open(tokenizer_path, 'r') as f:
        tokenizer_data = json.load(f)
    
    from tensorflow.keras.preprocessing.text import Tokenizer
    char_tokenizer = Tokenizer(char_level=True, lower=True, filters='')
    char_tokenizer.word_index = tokenizer_data['word_index']
    
    return char_tokenizer


def load_keras_model_and_preprocessor(model_path, preprocessor_path):
    """Load a Keras model and its preprocessor (without tokenizer)."""
    # Load model (compile=False for faster loading since we only need inference)
    model = keras.models.load_model(model_path, compile=False)
    
    # Load preprocessor dict (contains 'preprocessor' and 'scaler')
    with open(preprocessor_path, 'rb') as f:
        preprocessor_dict = pickle.load(f)
    
    return model, preprocessor_dict


def load_sklearn_pipeline(pipeline_path):
    """Load a sklearn pipeline."""
    with open(pipeline_path, 'rb') as f:
        pipeline = pickle.load(f)
    return pipeline


def load_transformer_model(model_path, tokenizer_name):
    """Load a transformer model and tokenizer."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    
    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()
    
    return model, tokenizer, device


def predict_keras_model(model, preprocessor_dict, char_tokenizer, data, maxlen=1000):
    """Make predictions using Keras hybrid model (CharCNN + MLP)."""
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    
    # Extract preprocessor and scaler
    preprocessor = preprocessor_dict['preprocessor']
    scaler = preprocessor_dict['scaler']
    
    # Process MLP features (TF-IDF + structured features)
    X_mlp_processed = preprocessor.transform(data)
    X_mlp_scaled = scaler.transform(X_mlp_processed)
    
    # Process CharCNN features (character sequences)
    texts = data['text'].astype(str).values
    X_char_seq = char_tokenizer.texts_to_sequences(texts)
    X_char_pad = pad_sequences(X_char_seq, maxlen=maxlen, padding='post', truncating='post')
    
    # Model expects [char_input, mlp_input]
    predictions = model.predict([X_char_pad, X_mlp_scaled], verbose=0).flatten()
    
    return predictions


def predict_sklearn_pipeline(pipeline, data, model_name=''):
    """Make predictions using sklearn pipeline."""
    # For Ridge model (text-only), extract just the text column
    # For XGBoost model (all features), use full dataframe
    if 'ridge' in model_name.lower():
        # Ridge expects only text data
        if isinstance(data, pd.DataFrame) and 'text' in data.columns:
            data_to_predict = data['text'].values
        else:
            raise ValueError(f"Ridge model requires DataFrame with 'text' column, got {type(data)}")
    else:
        # XGBoost and other models expect full dataframe
        data_to_predict = data.reset_index(drop=True) if isinstance(data, pd.DataFrame) else data
    
    # Check if pipeline has predict_proba (most classifiers)
    if hasattr(pipeline, 'predict_proba'):
        predictions = pipeline.predict_proba(data_to_predict)[:, 1]
    # Otherwise use decision_function (e.g., RidgeClassifier, SVM)
    elif hasattr(pipeline, 'decision_function'):
        # decision_function returns raw scores, need to convert to probabilities
        decision_scores = pipeline.decision_function(data_to_predict)
        predictions = expit(decision_scores)  # Sigmoid to convert to [0, 1]
    else:
        raise AttributeError(f"Pipeline {pipeline} has neither predict_proba nor decision_function")
    
    # Verify shape
    expected_len = len(data)
    if len(predictions) != expected_len:
        raise ValueError(f"Prediction length mismatch for {model_name}: expected {expected_len}, got {len(predictions)}")
    
    return predictions


def predict_transformer_model(model, tokenizer, device, data, max_length=512, batch_size=32):
    """Make predictions using transformer model."""
    texts = data['text'].astype(str).values
    predictions = []
    
    # Process in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize
        encoded = tokenizer(
            batch_texts.tolist(),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        # Move to device
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        
        # Get predictions
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[:, 1]  # Probability of positive class
            predictions.extend(probs.cpu().numpy())
    
    return np.array(predictions)

✓ Applied sklearn compatibility fix for _RemainderColsList


In [ ]:
# Initialize arrays to store out-of-fold predictions (for evaluation)
oof_predictions = {
    'hybrid_charcnn_mlp': np.zeros(len(train)),
    'xgboost_optuna': np.zeros(len(train)),
    'ridge': np.zeros(len(train)),
    'deberta': np.zeros(len(train))
}

# Initialize arrays to store test predictions (averaged across folds)
test_predictions = {
    'hybrid_charcnn_mlp': np.zeros(len(test)),
    'xgboost_optuna': np.zeros(len(test)),
    'ridge': np.zeros(len(test)),
    'deberta': np.zeros(len(test))
}

print("Generating out-of-fold and test predictions from base models...")
print("="*60)

# Pre-load character tokenizer once (shared across all hybrid model folds)
char_tokenizer = None
for model_name, config in BASE_MODELS.items():
    if config['type'] == 'keras' and 'tokenizer_file' in config:
        tokenizer_path = os.path.join(config['path'], config['tokenizer_file'])
        print(f"\nLoading shared character tokenizer from {config['path']}...")
        char_tokenizer = load_char_tokenizer(tokenizer_path)
        print(f"✓ Tokenizer loaded (vocab size: {len(char_tokenizer.word_index)})")
        break

print()

# Iterate through folds
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train, y), 1):
    print(f"\nProcessing Fold {fold_idx}/{N_FOLDS}")
    print(f"  Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
    # Get validation data for this fold
    val_data = train.iloc[val_idx].drop(['id', 'label'], axis=1)
    test_data = test.drop(['id'], axis=1)
    
    # Process each base model
    for model_name, config in BASE_MODELS.items():
        print(f"  Loading {model_name}...", end=" ", flush=True)
        
        if config['type'] == 'keras':
            # Load Keras model
            model_path = os.path.join(
                config['path'], 
                f"{config['model_prefix']}{fold_idx}{config['model_suffix']}"
            )
            preprocessor_path = os.path.join(
                config['path'],
                f"{config['preprocessor_prefix']}{fold_idx}{config['preprocessor_suffix']}"
            )
            
            # Load model and preprocessor (tokenizer already loaded)
            model, preprocessor_dict = load_keras_model_and_preprocessor(
                model_path, preprocessor_path
            )
            
            # Generate predictions
            val_preds = predict_keras_model(model, preprocessor_dict, char_tokenizer, val_data)
            test_preds = predict_keras_model(model, preprocessor_dict, char_tokenizer, test_data)
            
            # Clean up to free memory
            del model
            keras.backend.clear_session()
            
        elif config['type'] == 'sklearn':
            # Load sklearn pipeline
            pipeline_path = os.path.join(
                config['path'],
                f"{config['pipeline_prefix']}{fold_idx}{config['pipeline_suffix']}"
            )
            pipeline = load_sklearn_pipeline(pipeline_path)
            
            # Generate predictions (pass model_name to handle different input types)
            val_preds = predict_sklearn_pipeline(pipeline, val_data, model_name)
            test_preds = predict_sklearn_pipeline(pipeline, test_data, model_name)
        
        elif config['type'] == 'transformer':
            # Load transformer model
            model_path = os.path.join(config['path'], f"{config['model_prefix']}{fold_idx}")
            model, tokenizer, device = load_transformer_model(model_path, config['tokenizer_name'])
            
            # Generate predictions
            val_preds = predict_transformer_model(
                model, tokenizer, device, val_data, 
                max_length=config['max_length'], 
                batch_size=config['batch_size']
            )
            test_preds = predict_transformer_model(
                model, tokenizer, device, test_data, 
                max_length=config['max_length'], 
                batch_size=config['batch_size']
            )
            
            # Clean up to free memory
            del model, tokenizer
            torch.cuda.empty_cache()
        
        # Store out-of-fold predictions
        oof_predictions[model_name][val_idx] = val_preds
        
        # Accumulate test predictions (will average later)
        test_predictions[model_name] += test_preds / N_FOLDS
        
        print(f"✓ (val: {len(val_preds)}, test: {len(test_preds)})")

print("\n" + "="*60)
print("Base model prediction generation complete!")
print("\nOut-of-fold prediction shapes:")
for model_name, preds in oof_predictions.items():
    print(f"  {model_name:20s}: {preds.shape}")
print("\nTest prediction shapes:")
for model_name, preds in test_predictions.items():
    print(f"  {model_name:20s}: {preds.shape}")

Generating out-of-fold and test predictions from base models...

Loading shared character tokenizer from experiments_old/exp_8_hybrid_charcnn_mlp...
✓ Tokenizer loaded (vocab size: 772)


Processing Fold 1/5
  Train size: 3369, Val size: 843
  Loading deberta... 

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

## 5. Evaluate Base Model Performance

Let's check the performance of each base model on the training data using their out-of-fold predictions.

In [ ]:
print("Base Model Performance (Out-of-Fold):")
print("="*60)

base_model_oof_metrics = {}

for model_name, preds in oof_predictions.items():
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y, (preds >= 0.5).astype(int)),
        'precision': precision_score(y, (preds >= 0.5).astype(int)),
        'recall': recall_score(y, (preds >= 0.5).astype(int)),
        'f1': f1_score(y, (preds >= 0.5).astype(int)),
        'roc_auc': roc_auc_score(y, preds),
        'log_loss': log_loss(y, preds)
    }
    
    base_model_oof_metrics[model_name] = metrics
    
    print(f"\n{model_name}:")
    for metric, value in metrics.items():
        print(f"  {metric:12s}: {value:.6f}")

print("\n" + "="*60)

In [ ]:
# Create directory for Optuna plots
optuna_plots_dir = os.path.join(ENSEMBLE_DIR, 'optuna_plots')
os.makedirs(optuna_plots_dir, exist_ok=True)

# Plot optimization history
fig1 = plot_optimization_history(study)
fig1.write_html(os.path.join(optuna_plots_dir, 'optimization_history.html'))
print(f"Optimization history saved to: {optuna_plots_dir}/optimization_history.html")

# Plot parameter importances
fig2 = plot_param_importances(study)
fig2.write_html(os.path.join(optuna_plots_dir, 'param_importances.html'))
print(f"Parameter importances saved to: {optuna_plots_dir}/param_importances.html")

# Plot parallel coordinate
fig3 = plot_parallel_coordinate(study)
fig3.write_html(os.path.join(optuna_plots_dir, 'parallel_coordinate.html'))
print(f"Parallel coordinate saved to: {optuna_plots_dir}/parallel_coordinate.html")

print("\nOptuna visualizations saved successfully!")
print("Open the HTML files in a browser to view the plots.")

## 6. Simple Average Ensemble

Create a simple average ensemble of all base model predictions.

In [ ]:
# Simple average of all base model predictions
avg_oof_predictions = np.mean([
    oof_predictions['hybrid_charcnn_mlp'],
    oof_predictions['xgboost_optuna'],
    oof_predictions['ridge'],
    oof_predictions['deberta']
], axis=0)

avg_test_predictions = np.mean([
    test_predictions['hybrid_charcnn_mlp'],
    test_predictions['xgboost_optuna'],
    test_predictions['ridge'],
    test_predictions['deberta']
], axis=0)

# Calculate metrics for average ensemble
avg_metrics = {
    'accuracy': accuracy_score(y, (avg_oof_predictions >= 0.5).astype(int)),
    'precision': precision_score(y, (avg_oof_predictions >= 0.5).astype(int)),
    'recall': recall_score(y, (avg_oof_predictions >= 0.5).astype(int)),
    'f1': f1_score(y, (avg_oof_predictions >= 0.5).astype(int)),
    'roc_auc': roc_auc_score(y, avg_oof_predictions),
    'log_loss': log_loss(y, avg_oof_predictions)
}

print("\nSimple Average Ensemble Performance:")
print("="*60)
for metric, value in avg_metrics.items():
    print(f"  {metric:12s}: {value:.6f}")

## 7. Performance Comparison

In [ ]:
# Create comparison dataframe
comparison_data = []

# Base models
for model_name, metrics in base_model_oof_metrics.items():
    comparison_data.append({
        'Model': model_name,
        'Type': 'Base Model',
        'ROC-AUC': metrics['roc_auc'],
        'Log Loss': metrics['log_loss'],
        'F1 Score': metrics['f1'],
        'Accuracy': metrics['accuracy']
    })

# Simple average ensemble
comparison_data.append({
    'Model': 'Simple Average',
    'Type': 'Ensemble',
    'ROC-AUC': avg_metrics['roc_auc'],
    'Log Loss': avg_metrics['log_loss'],
    'F1 Score': avg_metrics['f1'],
    'Accuracy': avg_metrics['accuracy']
})

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False)

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Identify best model
best_model_idx = comparison_df['ROC-AUC'].idxmax()
best_model = comparison_df.loc[best_model_idx, 'Model']
best_roc_auc = comparison_df.loc[best_model_idx, 'ROC-AUC']

print(f"\nBest Model: {best_model} (ROC-AUC: {best_roc_auc:.6f})")

# Calculate improvement from ensemble
best_base_model = comparison_df[comparison_df['Type'] == 'Base Model'].iloc[0]
ensemble_model = comparison_df[comparison_df['Type'] == 'Ensemble'].iloc[0]

improvement = ensemble_model['ROC-AUC'] - best_base_model['ROC-AUC']
print(f"\nEnsemble Improvement over Best Base Model:")
print(f"  ROC-AUC: {improvement:+.6f}")
print(f"  Log Loss: {best_base_model['Log Loss'] - ensemble_model['Log Loss']:+.6f}")

## 8. Generate Submission File

In [ ]:
# Create submission for simple average ensemble
submission = pd.DataFrame({
    'id': test_ids,
    'probability': avg_test_predictions
})
submission_path = os.path.join(ENSEMBLE_DIR, f'{ENSEMBLE_NAME}_submission.csv')
submission.to_csv(submission_path, index=False)
print(f"Submission saved to: {submission_path}")

# Display sample
print("\nSample predictions:")
print(submission.head(10))

# Display statistics
print("\nPrediction statistics:")
print(f"  Mean: {avg_test_predictions.mean():.6f}")
print(f"  Std:  {avg_test_predictions.std():.6f}")
print(f"  Min:  {avg_test_predictions.min():.6f}")
print(f"  Max:  {avg_test_predictions.max():.6f}")

## 9. Save Metrics (Following Model Tracking Guide)

In [ ]:
# Create comprehensive metrics JSON
ensemble_metrics = {
    "experiment_name": ENSEMBLE_NAME,
    "model_type": "Simple Average Ensemble",
    "ensemble_method": "Simple averaging of base model predictions",
    "base_models": [
        {
            "name": "exp_8_hybrid_charcnn_mlp",
            "type": "Hybrid CharCNN + MLP",
            "oof_roc_auc": base_model_oof_metrics['hybrid_charcnn_mlp']['roc_auc']
        },
        {
            "name": "exp_4_all_xgboost_optuna", 
            "type": "XGBoost (Optuna)",
            "oof_roc_auc": base_model_oof_metrics['xgboost_optuna']['roc_auc']
        },
        {
            "name": "exp_2_tfidf_ridge",
            "type": "Ridge Classifier",
            "oof_roc_auc": base_model_oof_metrics['ridge']['roc_auc']
        },
        {
            "name": "exp_12_deberta",
            "type": "DeBERTa-v3",
            "oof_roc_auc": base_model_oof_metrics['deberta']['roc_auc']
        }
    ],
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "n_folds": N_FOLDS,
    "random_state": RANDOM_STATE,
    
    # Ensemble metrics
    "ensemble_metrics": avg_metrics,
    
    # Comparison with base models
    "comparison": {
        "best_base_model": {
            "name": comparison_df[comparison_df['Type'] == 'Base Model'].iloc[0]['Model'],
            "roc_auc": comparison_df[comparison_df['Type'] == 'Base Model'].iloc[0]['ROC-AUC']
        },
        "improvement": {
            "roc_auc": avg_metrics['roc_auc'] - comparison_df[comparison_df['Type'] == 'Base Model'].iloc[0]['ROC-AUC'],
            "log_loss": comparison_df[comparison_df['Type'] == 'Base Model'].iloc[0]['Log Loss'] - avg_metrics['log_loss']
        }
    }
}

# Save metrics
metrics_path = os.path.join(ENSEMBLE_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(ensemble_metrics, f, indent=2)

print(f"Metrics saved to: {metrics_path}")

## 10. Save Out-of-Fold Predictions

In [ ]:
# Save out-of-fold predictions from base models and ensemble
oof_predictions_df = pd.DataFrame({
    'id': train['id'].values,
    'label': y,
    'hybrid_charcnn_mlp': oof_predictions['hybrid_charcnn_mlp'],
    'xgboost_optuna': oof_predictions['xgboost_optuna'],
    'ridge': oof_predictions['ridge'],
    'deberta': oof_predictions['deberta'],
    'simple_average': avg_oof_predictions
})

oof_path = os.path.join(ENSEMBLE_DIR, 'oof_predictions.csv')
oof_predictions_df.to_csv(oof_path, index=False)
print(f"Out-of-fold predictions saved to: {oof_path}")

# Display sample
print("\nSample OOF predictions:")
print(oof_predictions_df.head(10))

# Display correlations
print("\nCorrelation between models:")
corr_matrix = oof_predictions_df[['hybrid_charcnn_mlp', 'xgboost_optuna', 'ridge', 'deberta']].corr()
print(corr_matrix)

## 11. Final Summary

In [ ]:
print("\n" + "="*80)
print("ENSEMBLE MODEL SUMMARY")
print("="*80)

print("\nBase Models:")
for i, model_info in enumerate(ensemble_metrics['base_models'], 1):
    print(f"  {i}. {model_info['name']}")
    print(f"     Type: {model_info['type']}")
    print(f"     OOF ROC-AUC: {model_info['oof_roc_auc']:.6f}")

print("\n" + "-"*80)
print("ENSEMBLE METHOD:")
print("-"*80)

print("\nSimple Average Ensemble:")
print(f"   Method: Equal-weight averaging of all base models")
print(f"   ROC-AUC:  {avg_metrics['roc_auc']:.6f}")
print(f"   Log Loss: {avg_metrics['log_loss']:.6f}")
print(f"   F1 Score: {avg_metrics['f1']:.6f}")
print(f"   Accuracy: {avg_metrics['accuracy']:.6f}")

print(f"\n   Improvement over Best Base Model:")
print(f"     ROC-AUC: {ensemble_metrics['comparison']['improvement']['roc_auc']:+.6f}")
print(f"     Log Loss: {ensemble_metrics['comparison']['improvement']['log_loss']:+.6f}")

print("\n" + "-"*80)
print("SAVED ARTIFACTS:")
print("-"*80)
print(f"  1. Metrics: {metrics_path}")
print(f"  2. Submission: {submission_path}")
print(f"  3. OOF Predictions: {oof_path}")

print("\n" + "="*80)
print("ENSEMBLE MODEL COMPLETE!")
print("="*80)